# Big Dataset Comparison: ML vs DL on Tabular Data

**Dataset:** Covertype - 581,012 rows x 54 features, 7-class classification of forest cover types.

**What this notebook does:**
1. Loads the dataset and creates one fixed train / val / test split with stratification.
2. Optionally caps the training fold at a fixed stratified subset so heavy DL models stay tractable on a laptop. **The same subset is used by every model**, so the comparison stays fair.
3. Fits one shared `StandardScaler` on that training fold and reuses it everywhere.
4. Trains four models on the same scaled data:
   - **Logistic Regression** - linear ML baseline.
   - **XGBoost** - gradient-boosted trees, the strong tabular ML model the project requirement calls out.
   - **MLP (sklearn)** - basic feed-forward network.
   - **TabNet** - attention-based DL architecture designed for tabular data, the proper DL counterpart to XGBoost.
5. Reports accuracy, macro precision / recall / F1, OvR macro ROC-AUC, training time, and inference time, plus confusion matrices and ROC curves.

**Why this dataset matters for the comparison:** with ~580k rows it's where we expect deep learning approaches to *finally* have enough data to compete with - or beat - gradient boosting. That contrast with the small-dataset notebook is the whole point of the project.

In [ ]:
try:
    import xgboost  # noqa: F401
except ImportError:
    %pip install -q xgboost
try:
    import pytorch_tabnet  # noqa: F401
except ImportError:
    %pip install -q pytorch-tabnet

In [ ]:
import warnings
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score, roc_curve,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.linear_model import LogisticRegression

from xgboost import XGBClassifier
from pytorch_tabnet.tab_model import TabNetClassifier
import torch

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
DATA_CANDIDATES = [
    PROJECT_ROOT / "dataset" / "datasetBig" / "covtype.csv",
    PROJECT_ROOT / "Data Set big" / "covtype.csv",
    PROJECT_ROOT / "Data set big" / "covtype.csv",
    PROJECT_ROOT / "datasetBig" / "covtype.csv",
]


def resolve_existing_path(candidates):
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        "Could not find covtype.csv. Checked: " + ", ".join(str(p) for p in candidates)
    )


DATA_PATH = resolve_existing_path(DATA_CANDIDATES)
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "big_dataset"
FIGURES_DIR = ARTIFACTS_DIR / "figures"
MODELS_DIR = ARTIFACTS_DIR / "models"
METRICS_DIR = ARTIFACTS_DIR / "metrics"

for folder in [ARTIFACTS_DIR, FIGURES_DIR, MODELS_DIR, METRICS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# README guidance allows fixed stratified subsets for heavy training. The same
# subset is consumed by EVERY model below, so the comparison stays fair.
USE_TRAIN_SUBSET_FOR_MODELING = True
MAX_TRAIN_SAMPLES = 120000
N_INFERENCE_REPEATS = 20

print(f"Project root: {PROJECT_ROOT}")
print(f"Using dataset: {DATA_PATH}")
print(f"Artifacts folder: {ARTIFACTS_DIR}")
print(f"Torch device available: {'cuda' if torch.cuda.is_available() else 'cpu'}")

## 1) Load the dataset

The CSV has 54 feature columns plus a `Cover_Type` target with values 1..7. We split into a feature matrix and target right after loading.

In [ ]:
df = pd.read_csv(DATA_PATH)

empty_cols = [c for c in df.columns if str(c).lower().startswith("unnamed")]
if empty_cols:
    df = df.drop(columns=empty_cols)

TARGET_COL = "Cover_Type" if "Cover_Type" in df.columns else df.columns[-1]
X_raw = df.drop(columns=[TARGET_COL]).copy()
y_raw = df[TARGET_COL].copy()
feature_cols = X_raw.columns.tolist()

display(df.head())
print(f"Shape: {df.shape}")
print(f"Target column: {TARGET_COL}")
print(f"Feature count: {len(feature_cols)}")

## 2) Validate and re-index the target

We confirm there are no missing values and remap the original 1..7 class labels to 0..6 - that's what scikit-learn, XGBoost, and TabNet all expect for multiclass classification. The `index_to_class` dict lets us flip back to the original labels for plots.

In [ ]:
missing_values = df.isna().sum().sum()
duplicate_rows = df.duplicated().sum()

print(f"Total missing values: {missing_values}")
print(f"Duplicate rows: {duplicate_rows}")

X = X_raw.apply(pd.to_numeric, errors="raise")
y_numeric = pd.to_numeric(y_raw, errors="raise").astype(int)

raw_classes = np.sort(y_numeric.unique())
class_to_index = {cls: idx for idx, cls in enumerate(raw_classes)}
index_to_class = {idx: int(cls) for cls, idx in class_to_index.items()}
y = y_numeric.map(class_to_index).astype(int)
N_CLASSES = len(raw_classes)

print(f"Class count: {N_CLASSES}")
print("Class mapping (internal -> original):", index_to_class)
print("Class balance (normalized):")
print(y.value_counts(normalize=True).sort_index())

## 3) Stratified split + tractable training subset

We do a 70 / 15 / 15 stratified split. The training fold has ~400k rows, which is fine for XGBoost but slow for the neural nets, so we draw a fixed stratified subset of `MAX_TRAIN_SAMPLES` rows once and reuse it for **every** model. Every model still validates on the full validation fold and is tested on the full test fold.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE,
)

val_ratio_within_train_val = 0.15 / 0.85
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_ratio_within_train_val,
    stratify=y_train_val, random_state=RANDOM_STATE,
)

X_train_model = X_train.copy()
y_train_model = y_train.copy()

if USE_TRAIN_SUBSET_FOR_MODELING and len(X_train_model) > MAX_TRAIN_SAMPLES:
    X_train_model, _, y_train_model, _ = train_test_split(
        X_train_model, y_train_model,
        train_size=MAX_TRAIN_SAMPLES,
        stratify=y_train_model, random_state=RANDOM_STATE,
    )

print("Split sizes (full):")
print(f"Train full: {len(X_train)}")
print(f"Validation: {len(X_val)}")
print(f"Test: {len(X_test)}")
print("\nRows used for model fitting:", len(X_train_model))

## 4) Shared preprocessing + metric helpers

One scaler fitted on the training subset, applied to val / test as well. The metric helpers handle the multiclass case (OvR macro ROC-AUC instead of plain binary AUC) and produce a single mean ROC curve across the seven classes for plotting.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_model)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

y_train_arr = y_train_model.to_numpy()
y_val_arr = y_val.to_numpy()
y_test_arr = y_test.to_numpy()


def get_class_scores(model, X_input):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X_input)
    raise ValueError(f"Model {type(model).__name__} does not expose predict_proba.")


def compute_common_metrics(y_true, y_pred, y_score):
    metrics = {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision_macro": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall_macro": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1_macro": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }
    if y_score.ndim == 2 and y_score.shape[1] > 2:
        metrics["roc_auc"] = float(
            roc_auc_score(y_true, y_score, multi_class="ovr", average="macro")
        )
    else:
        positive_score = y_score[:, 1] if y_score.ndim == 2 else y_score
        metrics["roc_auc"] = float(roc_auc_score(y_true, positive_score))
    return metrics


def compute_macro_roc_curve(y_true, y_score, n_classes):
    if n_classes <= 2:
        positive_score = y_score[:, 1] if y_score.ndim == 2 else y_score
        fpr, tpr, _ = roc_curve(y_true, positive_score)
        return fpr, tpr
    y_bin = label_binarize(y_true, classes=np.arange(n_classes))
    grid = np.linspace(0, 1, 400)
    mean_tpr = np.zeros_like(grid)
    for class_idx in range(n_classes):
        fpr_i, tpr_i, _ = roc_curve(y_bin[:, class_idx], y_score[:, class_idx])
        mean_tpr += np.interp(grid, fpr_i, tpr_i)
    mean_tpr /= n_classes
    return grid, mean_tpr


def evaluate_on_val(model):
    val_pred = model.predict(X_val_scaled)
    val_score = get_class_scores(model, X_val_scaled)
    return compute_common_metrics(y_val_arr, val_pred, val_score)

## 5) Train the four baseline models

One section per model, training time tracked, validation metrics collected into one shared list. Hyperparameters are reasonable defaults for a 7-class tabular problem of this size; we'll do a small search after this.

In [ ]:
baseline_results = []

logreg_baseline = LogisticRegression(
    C=1.0, solver="lbfgs", max_iter=300, random_state=RANDOM_STATE,
)
start = time.perf_counter()
logreg_baseline.fit(X_train_scaled, y_train_arr)
logreg_train_time = time.perf_counter() - start
logreg_val = evaluate_on_val(logreg_baseline)
baseline_results.append({"model": "LogisticRegression",
                         "train_time_sec": logreg_train_time,
                         **{f"val_{k}": v for k, v in logreg_val.items()}})
print("LogReg val:", logreg_val)

In [ ]:
xgb_baseline = XGBClassifier(
    n_estimators=400, max_depth=8, learning_rate=0.1,
    subsample=0.9, colsample_bytree=0.9,
    objective="multi:softprob", num_class=N_CLASSES, eval_metric="mlogloss",
    tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1,
)
start = time.perf_counter()
xgb_baseline.fit(X_train_scaled, y_train_arr)
xgb_train_time = time.perf_counter() - start
xgb_val = evaluate_on_val(xgb_baseline)
baseline_results.append({"model": "XGBoost",
                         "train_time_sec": xgb_train_time,
                         **{f"val_{k}": v for k, v in xgb_val.items()}})
print("XGBoost val:", xgb_val)

In [ ]:
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
    alpha=1e-4, learning_rate_init=1e-3, batch_size=512,
    max_iter=120, early_stopping=True, n_iter_no_change=10,
    random_state=RANDOM_STATE,
)
start = time.perf_counter()
mlp_baseline.fit(X_train_scaled, y_train_arr)
mlp_train_time = time.perf_counter() - start
mlp_val = evaluate_on_val(mlp_baseline)
baseline_results.append({"model": "MLPClassifier",
                         "train_time_sec": mlp_train_time,
                         **{f"val_{k}": v for k, v in mlp_val.items()}})
print("MLP val:", mlp_val)

In [ ]:
tabnet_baseline = TabNetClassifier(
    n_d=32, n_a=32, n_steps=5, gamma=1.5, lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size": 10, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type="entmax", seed=RANDOM_STATE, verbose=0,
)
start = time.perf_counter()
tabnet_baseline.fit(
    X_train=X_train_scaled, y_train=y_train_arr,
    eval_set=[(X_val_scaled, y_val_arr)],
    eval_metric=["accuracy"],
    max_epochs=80, patience=10,
    batch_size=4096, virtual_batch_size=512,
    drop_last=False,
)
tabnet_train_time = time.perf_counter() - start
tabnet_val = evaluate_on_val(tabnet_baseline)
baseline_results.append({"model": "TabNet",
                         "train_time_sec": tabnet_train_time,
                         **{f"val_{k}": v for k, v in tabnet_val.items()}})
print("TabNet val:", tabnet_val)

baseline_df = pd.DataFrame(baseline_results).set_index("model")
display(baseline_df.style.format("{:.4f}", subset=baseline_df.select_dtypes(float).columns))

## 6) Compact validation search

Same protocol as the small notebook: select the best config per family by validation macro-F1 (tie-break with ROC-AUC). Search spaces are intentionally compact because each fit on 120k rows takes a while - especially TabNet.

In [ ]:
def is_better(candidate, incumbent):
    if incumbent is None:
        return True
    if candidate["val_f1_macro"] > incumbent["val_f1_macro"]:
        return True
    if (candidate["val_f1_macro"] == incumbent["val_f1_macro"]
            and candidate["val_roc_auc"] > incumbent["val_roc_auc"]):
        return True
    return False


def run_sklearn_search(model_name, base_estimator, param_grid):
    records, best_record = [], None
    for params in ParameterGrid(param_grid):
        model = clone(base_estimator).set_params(**params)
        start = time.perf_counter()
        model.fit(X_train_scaled, y_train_arr)
        train_time_sec = time.perf_counter() - start
        val_metrics = evaluate_on_val(model)
        record = {"model_family": model_name, "params": params,
                  "train_time_sec": train_time_sec,
                  **{f"val_{k}": v for k, v in val_metrics.items()}}
        records.append(record)
        if is_better(record, best_record):
            best_record = record
    return pd.DataFrame(records), best_record


logreg_grid = {"C": [0.3, 1.0, 3.0], "solver": ["lbfgs"], "max_iter": [300]}
search_logreg_df, best_logreg_record = run_sklearn_search(
    "LogisticRegression", LogisticRegression(random_state=RANDOM_STATE), logreg_grid,
)

mlp_grid = {
    "hidden_layer_sizes": [(128, 64), (256, 128)],
    "alpha": [1e-4, 5e-4],
    "learning_rate_init": [1e-3],
    "max_iter": [120],
}
search_mlp_df, best_mlp_record = run_sklearn_search(
    "MLPClassifier",
    MLPClassifier(random_state=RANDOM_STATE, early_stopping=True,
                  n_iter_no_change=10, batch_size=512),
    mlp_grid,
)

xgb_configs = [
    {"n_estimators": 300, "max_depth": 6,  "learning_rate": 0.1},
    {"n_estimators": 400, "max_depth": 8,  "learning_rate": 0.1},
    {"n_estimators": 600, "max_depth": 8,  "learning_rate": 0.05},
]
xgb_records, best_xgb_record = [], None
for params in xgb_configs:
    full = {**params, "subsample": 0.9, "colsample_bytree": 0.9,
            "objective": "multi:softprob", "num_class": N_CLASSES,
            "eval_metric": "mlogloss", "tree_method": "hist",
            "random_state": RANDOM_STATE, "n_jobs": -1}
    model = XGBClassifier(**full)
    start = time.perf_counter()
    model.fit(X_train_scaled, y_train_arr)
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    record = {"model_family": "XGBoost", "params": params,
              "train_time_sec": train_time_sec,
              **{f"val_{k}": v for k, v in val_metrics.items()}}
    xgb_records.append(record)
    if is_better(record, best_xgb_record):
        best_xgb_record = record
search_xgb_df = pd.DataFrame(xgb_records)

tabnet_configs = [
    {"n_d": 16, "n_a": 16, "n_steps": 4, "gamma": 1.3, "lr": 2e-2},
    {"n_d": 32, "n_a": 32, "n_steps": 5, "gamma": 1.5, "lr": 2e-2},
]
tabnet_records, best_tabnet_record = [], None
for params in tabnet_configs:
    model = TabNetClassifier(
        n_d=params["n_d"], n_a=params["n_a"], n_steps=params["n_steps"],
        gamma=params["gamma"], lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=params["lr"]),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax", seed=RANDOM_STATE, verbose=0,
    )
    start = time.perf_counter()
    model.fit(
        X_train=X_train_scaled, y_train=y_train_arr,
        eval_set=[(X_val_scaled, y_val_arr)],
        eval_metric=["accuracy"],
        max_epochs=80, patience=10,
        batch_size=4096, virtual_batch_size=512,
        drop_last=False,
    )
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    record = {"model_family": "TabNet", "params": params,
              "train_time_sec": train_time_sec,
              **{f"val_{k}": v for k, v in val_metrics.items()}}
    tabnet_records.append(record)
    if is_better(record, best_tabnet_record):
        best_tabnet_record = record
search_tabnet_df = pd.DataFrame(tabnet_records)

search_results = pd.concat(
    [search_logreg_df, search_mlp_df, search_xgb_df, search_tabnet_df],
    ignore_index=True,
)
display(
    search_results[[
        "model_family", "val_accuracy", "val_precision_macro", "val_recall_macro",
        "val_f1_macro", "val_roc_auc", "train_time_sec", "params"
    ]].sort_values(["model_family", "val_f1_macro", "val_roc_auc"], ascending=[True, False, False])
)

print("Best per family:")
for rec in [best_logreg_record, best_xgb_record, best_mlp_record, best_tabnet_record]:
    print(f"  {rec['model_family']}: F1={rec['val_f1_macro']:.4f} AUC={rec['val_roc_auc']:.4f} params={rec['params']}")

## 7) Refit each winner and evaluate on the held-out test set

Same pattern as the small dataset. The test fold has been untouched since the very first split.

In [ ]:
def build_logreg(params):
    return LogisticRegression(random_state=RANDOM_STATE).set_params(**params)


def build_mlp(params):
    return MLPClassifier(
        random_state=RANDOM_STATE, early_stopping=True,
        n_iter_no_change=10, batch_size=512,
    ).set_params(**params)


def build_xgb(params):
    full = {**params, "subsample": 0.9, "colsample_bytree": 0.9,
            "objective": "multi:softprob", "num_class": N_CLASSES,
            "eval_metric": "mlogloss", "tree_method": "hist",
            "random_state": RANDOM_STATE, "n_jobs": -1}
    return XGBClassifier(**full)


def build_tabnet(params):
    return TabNetClassifier(
        n_d=params["n_d"], n_a=params["n_a"], n_steps=params["n_steps"],
        gamma=params["gamma"], lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=params["lr"]),
        scheduler_params={"step_size": 10, "gamma": 0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type="entmax", seed=RANDOM_STATE, verbose=0,
    )


def fit_tabnet(model):
    model.fit(
        X_train=X_train_scaled, y_train=y_train_arr,
        eval_set=[(X_val_scaled, y_val_arr)],
        eval_metric=["accuracy"], max_epochs=80, patience=10,
        batch_size=4096, virtual_batch_size=512, drop_last=False,
    )


selection_plan = {
    "LogisticRegression": (build_logreg, best_logreg_record["params"], "sklearn"),
    "XGBoost":            (build_xgb,    best_xgb_record["params"],    "sklearn"),
    "MLPClassifier":      (build_mlp,    best_mlp_record["params"],    "sklearn"),
    "TabNet":             (build_tabnet, best_tabnet_record["params"], "tabnet"),
}

selected_models = {}
selected_model_info = {}
for model_family, (builder, params, kind) in selection_plan.items():
    model = builder(params)
    start = time.perf_counter()
    if kind == "tabnet":
        fit_tabnet(model)
    else:
        model.fit(X_train_scaled, y_train_arr)
    train_time_sec = time.perf_counter() - start
    val_metrics = evaluate_on_val(model)
    selected_models[model_family] = model
    selected_model_info[model_family] = {
        "best_params": params, "train_time_sec": train_time_sec,
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

display(pd.DataFrame(selected_model_info).T)

In [ ]:
test_results = {}
model_predictions = {}
model_scores = {}
for model_family, model in selected_models.items():
    y_pred = model.predict(X_test_scaled)
    y_score = get_class_scores(model, X_test_scaled)
    test_results[model_family] = compute_common_metrics(y_test_arr, y_pred, y_score)
    model_predictions[model_family] = y_pred
    model_scores[model_family] = y_score

test_results_df = pd.DataFrame(test_results).T
display(test_results_df.style.format("{:.4f}"))

## 8) Confusion matrices and macro ROC curves

Confusion matrices use the original 1..7 class labels for readability. The ROC plot is the **macro-averaged one-vs-rest curve**, which is the standard way to summarise multiclass discrimination on a single chart.

In [ ]:
label_names = [str(index_to_class[i]) for i in sorted(index_to_class.keys())]
n_models = len(selected_models)
fig_cm, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 5))
if n_models == 1:
    axes = [axes]

for ax, (model_family, y_pred) in zip(axes, model_predictions.items()):
    cm = confusion_matrix(y_test_arr, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=label_names, yticklabels=label_names, ax=ax)
    ax.set_title(f"Confusion Matrix - {model_family}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

fig_roc, ax_roc = plt.subplots(figsize=(8, 6))
for model_family, y_score in model_scores.items():
    fpr, tpr = compute_macro_roc_curve(y_test_arr, y_score, N_CLASSES)
    auc_value = test_results[model_family]["roc_auc"]
    ax_roc.plot(fpr, tpr, label=f"{model_family} (macro AUC={auc_value:.4f})")
ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Chance")
ax_roc.set_title("Macro ROC Curve (OvR) on Test Set")
ax_roc.set_xlabel("False Positive Rate")
ax_roc.set_ylabel("True Positive Rate")
ax_roc.legend(loc="lower right")
ax_roc.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 9) Training and inference time benchmark

On a dataset this size, training time is a real cost. Expect TabNet and MLP to be the slowest, XGBoost to be moderate, and Logistic Regression to be the fastest. Inference is run a few times in a loop to get a stable per-sample timing.

In [ ]:
timing_rows = []
for model_family, model in selected_models.items():
    start_infer = time.perf_counter()
    for _ in range(N_INFERENCE_REPEATS):
        _ = model.predict(X_test_scaled)
    infer_total_sec = time.perf_counter() - start_infer
    timing_rows.append({
        "model_family": model_family,
        "train_time_sec": selected_model_info[model_family]["train_time_sec"],
        "inference_total_sec": infer_total_sec,
        "inference_per_sample_ms": (infer_total_sec / (N_INFERENCE_REPEATS * len(X_test_scaled))) * 1000,
    })
timing_df = pd.DataFrame(timing_rows).set_index("model_family")
display(timing_df.style.format("{:.6f}"))

## 10) Final comparison table and saved artifacts

The full side-by-side: validation metrics, test metrics, training time, inference time, and the chosen hyperparameters per model. Saved CSVs / JSON live under `artifacts/big_dataset/` and the trained models are written to disk for downstream use.

In [ ]:
final_rows = []
for model_family in selected_models.keys():
    info = selected_model_info[model_family]
    final_rows.append({
        "model_family": model_family,
        "val_accuracy": info["val_accuracy"],
        "val_precision_macro": info["val_precision_macro"],
        "val_recall_macro": info["val_recall_macro"],
        "val_f1_macro": info["val_f1_macro"],
        "val_roc_auc": info["val_roc_auc"],
        "test_accuracy": test_results[model_family]["accuracy"],
        "test_precision_macro": test_results[model_family]["precision_macro"],
        "test_recall_macro": test_results[model_family]["recall_macro"],
        "test_f1_macro": test_results[model_family]["f1_macro"],
        "test_roc_auc": test_results[model_family]["roc_auc"],
        "train_time_sec": timing_df.loc[model_family, "train_time_sec"],
        "inference_per_sample_ms": timing_df.loc[model_family, "inference_per_sample_ms"],
        "best_params": json.dumps(info["best_params"]),
    })

final_comparison_df = pd.DataFrame(final_rows).sort_values("test_f1_macro", ascending=False)
display(final_comparison_df)

comparison_csv_path = METRICS_DIR / "final_comparison_big_dataset.csv"
comparison_json_path = METRICS_DIR / "final_comparison_big_dataset.json"
search_csv_path = METRICS_DIR / "validation_search_results_big_dataset.csv"
coefficients_csv_path = METRICS_DIR / "logreg_coefficients_big_dataset.csv"

search_results_to_save = search_results.copy()
search_results_to_save["params"] = search_results_to_save["params"].apply(json.dumps)
search_results_to_save.to_csv(search_csv_path, index=False)
final_comparison_df.to_csv(comparison_csv_path, index=False)

with open(comparison_json_path, "w", encoding="utf-8") as f:
    json.dump(final_rows, f, indent=2)

if "LogisticRegression" in selected_models:
    logreg_model = selected_models["LogisticRegression"]
    coef_importance = np.abs(logreg_model.coef_).mean(axis=0)
    coef_df = pd.DataFrame({
        "feature": feature_cols,
        "mean_abs_coefficient": coef_importance,
    }).sort_values("mean_abs_coefficient", ascending=False)
    coef_df.to_csv(coefficients_csv_path, index=False)

for model_family, model in selected_models.items():
    if model_family == "TabNet":
        model.save_model(str(MODELS_DIR / "TabNet_best"))
    else:
        joblib.dump(model, MODELS_DIR / f"{model_family}_best.joblib")
joblib.dump(scaler, MODELS_DIR / "standard_scaler.joblib")

with open(METRICS_DIR / "feature_columns.json", "w", encoding="utf-8") as f:
    json.dump(feature_cols, f, indent=2)
with open(METRICS_DIR / "class_mapping.json", "w", encoding="utf-8") as f:
    json.dump(index_to_class, f, indent=2)

cm_plot_path = FIGURES_DIR / "confusion_matrices_test.png"
roc_plot_path = FIGURES_DIR / "roc_curves_test.png"
fig_cm.savefig(cm_plot_path, dpi=200, bbox_inches="tight")
fig_roc.savefig(roc_plot_path, dpi=200, bbox_inches="tight")

print("Saved artifacts:")
for p in [comparison_csv_path, comparison_json_path, search_csv_path,
         coefficients_csv_path, cm_plot_path, roc_plot_path,
         MODELS_DIR / "standard_scaler.joblib"]:
    print(f"- {p}")

## 11) Takeaways for the class

- **Same protocol as the small notebook** - same scaler, same train subset, same val / test splits across all four models. Differences in numbers reflect model capacity, not data.
- **Logistic Regression hits a ceiling fast.** A linear decision boundary cannot capture interactions like 'elevation AND soil type AND slope' that are clearly important here.
- **XGBoost is a strong, fast option.** It handles non-linear feature interactions natively via tree splits and trains in minutes on this scale.
- **MLP improves on LR** because non-linear hidden units finally let the model learn feature interactions.
- **TabNet is the deep-learning model designed for tabular data.** With 580k rows it has enough data to do interesting things - its sequential attention picks features step by step instead of relying on hand-crafted splits.
- **The big takeaway across both notebooks:** on the small dataset the simple ML model is hard to beat; on the large dataset gradient boosting and tabular DL pull meaningfully ahead. That's the headline answer to the question 'does dataset size change which model family wins?'